# A cell that `_pseudo_sync_runner` cannot finish haunts later cells

IPython runs a cell through `_pseudo_sync_runner` when `should_run_async()` returns `False`.
Shells that must run every cell synchronously override it to do so, e.g. a kernel that
marshals cells onto one particular thread.

If such a cell contains a top-level `await`, it still compiles into a coroutine. The runner
advances it once, sees that it is waiting for an event loop, and raises
`RuntimeError: 'run_cell_async' needs a real async loop` — but it leaves the coroutine
**suspended**.

The garbage collector closes it later, at whatever moment it happens to run. Closing throws
`GeneratorExit` into the coroutine, the bare `except:` in `run_code` and `run_ast_nodes`
catch it, and `showtraceback()` reports it as an error of **whichever cell is running then**.

This notebook reproduces that in a plain IPython kernel. Run it top to bottom.

In [ ]:
import IPython

IPython.__version__

'9.17.1'

## 1. Make the shell run every cell synchronously

We also record every exception `showtraceback()` reports, together with the cell that was
running at the time, so the attribution is easy to see.

In [2]:
import sys

ip = get_ipython()
ip.should_run_async = lambda *args, **kwargs: False

reports = []  # (cell number, exception type) for every showtraceback() call
_showtraceback = ip.showtraceback


def recording_showtraceback(*args, **kwargs):
    # execution_count has already moved on to the next cell by now.
    reports.append((ip.execution_count - 1, sys.exc_info()[0].__name__))
    return _showtraceback(*args, **kwargs)


ip.showtraceback = recording_showtraceback

## 2. Run a cell with a top-level `await`

This fails, as it should: there is no event loop to run it on.

In [3]:
import asyncio

await asyncio.sleep(0)

RuntimeError: 'run_cell_async' needs a real async loop

Its coroutine is now suspended rather than closed. It stays alive for now because
`sys.last_value` (set by `showtraceback()`) still refers to the `RuntimeError`, whose traceback
refers to the runner's frame and through it to the coroutine.

## 3. Any later error releases it

This `ZeroDivisionError` replaces `sys.last_value`, so nothing refers to the suspended coroutine
any more — except a reference cycle, which only the cyclic garbage collector can break.

In [4]:
1 / 0

ZeroDivisionError: division by zero

## 4. A harmless cell, run when the garbage collector closes the coroutine

This cell does nothing that can fail. Here the collection is forced to make the timing
deterministic; normally the collector runs on its own, during any cell.

**Without the fix this cell shows `GeneratorExit` tracebacks** — one from `run_code` and one from
`run_ast_nodes`, as the exception unwinds through both.

In [5]:
import gc

gc.collect()
print("This cell only collects garbage.")

GeneratorExit: 

GeneratorExit: 

This cell only collects garbage.


## 5. What `showtraceback()` reported, and for which cell

The `GeneratorExit` entries belong to the `await` cell, but they are attributed to the
garbage-collecting cell. In a kernel they are also published on iopub under that cell's
request, and the last traceback replaces that cell's own one in its `execute_reply`.

In [6]:
reports

[(3, 'RuntimeError'),
 (4, 'ZeroDivisionError'),
 (5, 'GeneratorExit'),
 (5, 'GeneratorExit')]

## With the fix

`_pseudo_sync_runner` closes the coroutine before raising, and `run_code` / `run_ast_nodes` let
the `GeneratorExit` of an async cell unwind instead of reporting it. The `await` cell still fails
with the same `RuntimeError`, nothing is left suspended, the garbage-collecting cell prints its
message without errors, and `reports` only contains the `RuntimeError` and the
`ZeroDivisionError`.

## Clean up

In [7]:
del ip.should_run_async, ip.showtraceback